# GPU embedding pipeline — Colab

Runs **only the stages that need a GPU** and hands back small artifacts:

| stage | CPU (local) | T4 (here) | artifact |
|---|---|---|---|
| YOLO detect+embed crops | ~1.5 h / scene | ~3 min | `crop_embeddings.pt` (9 MB) |
| EigenPlaces frame descriptors | ~2 h / scene | ~2 min | `crop_embeddings_eigenplaces.pt` (20 MB) |
| DINOv2 crop keys | ~1 h / scene | ~2 min | `crop_embeddings_dinov2.pt` (14 MB) |
| CLIP label verification | ~40 min / scene | ~1 min | `crop_clip_verify.pt` (small) |
| Replica GT extraction | minutes | minutes | `gt_instances.json` (KBs) |

Everything downstream (relational batteries, instance recall, merge
comparisons, cross-traverse recall, inspectors) is numpy and runs in
**seconds locally** — do not move it here.

Order: run cells top to bottom — and again from the top after any
runtime restart. Outputs persist in Drive and every stage skips work
that already exists; a fresh VM only re-runs the scene fetch (minutes).
Progress is mirrored to `Drive/ssnslam_colab/workspace/run_log_embeds.txt`,
readable from any device without touching this page.


## 0 · GPU check + Drive mount

Runtime → Change runtime type → **T4 GPU** before running this.


In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True,
                     text=True).stdout or 'NO GPU — set Runtime > T4')
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/ssnslam_colab')
WORKSPACE = DRIVE / 'workspace'      # repo + raw scene data (big)
ARTIFACTS = DRIVE / 'artifacts'      # the small files you take home
for p in (WORKSPACE, ARTIFACTS):
    p.mkdir(parents=True, exist_ok=True)
print('workspace:', WORKSPACE)


## 1 · Clone the repo into Drive + install

Cloning into Drive means the next session skips this entirely.


In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/SynapticScotsman/Semantic-Spiking-Neural-SLAM-2023.git'
BRANCH = 'results-sites'
REPO_DIR = WORKSPACE / 'Semantic-Spiking-Neural-SLAM-2023'

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '50',
                    REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH],
                   check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH],
                   check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard',
                    f'origin/{BRANCH}'], check=True)
os.chdir(REPO_DIR)
print('at', os.getcwd())
print(subprocess.run(['git', 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout)


In [ ]:
# Colab already ships torch/torchvision — only the extras are needed.
!pip -q install ultralytics transformers pillow numpy scipy 2>&1 | tail -2


## 2 · Choose the work

`SCENES` — Replica scene ids. `STAGES` — comment out what you don't need.

A full 8-scene Replica sweep (fetch + all stages) is roughly 2–3 h here
against ~20 h on CPU, and most of that is the *download*, not the GPU.


In [ ]:
SCENES = ['room0']            # e.g. ['room0','room1','room2','office0']

STAGES = [
    'fetch',        # RGB-D + poses (range-fetch, resumable)
    'gt',           # GT instance centroids from the vMAP renders
    'yolo',         # detections + crop embeddings
    'eigenplaces',  # place descriptors (localisation frontend)
    'dinov2',       # crop keys, small (object/instance frontend)
    'clip',         # CLIP label verification cache
    # 'dinov2-large',  # <- the QUEUED instance-key rerun: 1024-d keys
    #                  #    for instance_recall.py. ~50k crops across 5
    #                  #    scenes is hours on CPU, ~10 min on a T4.
]

# outputs (small, valuable) persist in Drive. Raw scene data goes on
# the fast LOCAL VM disk instead — Drive-FUSE reads make YOLO several
# times slower, and the resumable fetch re-pulls a scene in minutes on
# Colab bandwidth, so persistence buys less than speed here.
import os
outs = WORKSPACE / 'outputs'
outs.mkdir(parents=True, exist_ok=True)
if not os.path.islink('outputs') and not os.path.exists('outputs'):
    os.symlink(outs, 'outputs')
local = Path('/content/replica_data')
local.mkdir(parents=True, exist_ok=True)
os.makedirs('data', exist_ok=True)
# older sessions symlinked data/replica into Drive — re-point to local disk
if os.path.islink('data/replica') and '/drive/' in os.path.realpath('data/replica'):
    os.remove('data/replica')
if not os.path.islink('data/replica') and not os.path.exists('data/replica'):
    os.symlink(local, 'data/replica')
print('data ->', os.path.realpath('data/replica'), '(local VM disk)')
print('outputs ->', os.path.realpath('outputs'), '(Drive)')


## 3 · Run the pipeline

Each stage is the *same command you run locally* — no Colab-specific
code paths, so results are directly comparable. Stages that already
have their artifact are skipped.


In [ ]:
import subprocess, time, os, datetime, threading
from pathlib import Path

# A runtime restart resets the working directory but keeps this notebook
# open — guard against running this cell outside the repo.
assert os.path.exists('vsa_cognitive_mapping'), (
    'Kernel is not inside the repo (runtime was restarted?). '
    'Run cells 0-2 first — cell 1 clones/updates the repo and chdirs into it.')
assert 'WORKSPACE' in globals(), 'run cell 0 first (Drive mount defines WORKSPACE)'

# Output streams here AND mirrors to a Drive log — checkable from
# drive.google.com on any device. A failed stage raises immediately;
# the … heartbeat proves a quiet stage is still alive.
LOG = f'{WORKSPACE}/run_log_embeds.txt'
def _log(line):
    with open(LOG, 'a') as f: f.write(line + chr(10))
def sh(cmd):
    c = ' '.join(map(str, cmd))
    stamp = datetime.datetime.now().strftime('%H:%M:%S')
    print(f'[{stamp}] $ {c}', flush=True); _log(f'[{stamp}] $ {c}')
    t0 = time.time(); last = [t0]
    env = dict(os.environ, PYTHONUNBUFFERED='1')
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, errors='replace', env=env)
    def pulse():
        while p.poll() is None:
            time.sleep(30)
            if p.poll() is None and time.time() - last[0] > 300:
                m = int((time.time() - t0) / 60)
                msg = f'  … still running ({m} min elapsed, no new output — normal for long model passes)'
                print(msg, flush=True); _log(msg); last[0] = time.time()
    threading.Thread(target=pulse, daemon=True).start()
    with open(LOG, 'a') as f:
        for ln in p.stdout:
            ln = ln.rstrip()
            print(ln, flush=True); f.write(ln + chr(10)); f.flush(); last[0] = time.time()
    rc = p.wait()
    done = f'  exit {rc} in {time.time()-t0:.0f}s'
    print(done, flush=True); _log(done)
    if rc != 0:
        raise RuntimeError(f'stage failed: {c} — full output in {LOG}; later stages for this scene depend on it')
    return True

def cfg(s):
    return f'vsa_cognitive_mapping/configs/replica_{s}.json'

for s in SCENES:
    out = Path(f'outputs/replica_{s}')
    print(f'\n=========== {s} ===========', flush=True)

    if 'fetch' in STAGES and not Path(f'data/replica/{s}/poses.csv').exists():
        sh(['python', 'tools/prepare_replica.py', '--scene', s])

    if 'gt' in STAGES and not (out / 'gt_instances.json').exists():
        sh(['python', 'tools/replica_gt_from_renders.py', '--scene', s])

    if 'yolo' in STAGES and not (out / 'crop_embeddings.pt').exists():
        sh(['python', '-m', 'vsa_cognitive_mapping.classroom_pipeline',
            'embed-crops', '--dataset', cfg(s)])

    if 'eigenplaces' in STAGES and not (out / 'crop_embeddings_eigenplaces.pt').exists():
        sh(['python', '-m', 'vsa_cognitive_mapping.vpr_frontend',
            '--dataset', cfg(s), '--model', 'eigenplaces', '--batch', '32'])

    if 'dinov2' in STAGES and not (out / 'crop_embeddings_dinov2.pt').exists():
        sh(['python', '-m', 'vsa_cognitive_mapping.encoder_comparison',
            '--dataset', cfg(s), '--encoders', 'dinov2'])

    if 'dinov2-large' in STAGES and not (out / 'crop_embeddings_dinov2-large.pt').exists():
        sh(['python', '-m', 'vsa_cognitive_mapping.encoder_comparison',
            '--dataset', cfg(s), '--encoders', 'dinov2:large',
            '--batch', '64'])

    if 'clip' in STAGES and not (out / 'crop_clip_verify.pt').exists():
        sh(['python', '-m', 'vsa_cognitive_mapping.object_grounding',
            '--dataset', cfg(s), '--gt-json', str(out / 'gt_instances.json'),
            '--verify-crops', '--max-per-class', '60'])

print('\nall requested stages done')


## 4 · Pack the small artifacts to take home

Only the outputs — never the raw scene data. Expect ~30–50 MB per scene.
This cell REFUSES to produce an empty archive: if a scene has no outputs
yet it says so and, if nothing at all was packed, it raises instead of
handing you a 22-byte zip.


In [ ]:
import zipfile, os, time
from pathlib import Path

assert os.path.exists('vsa_cognitive_mapping'), 'run cells 0-2 first (runtime restart lost the working directory)'

KEEP = ('.pt', '.json', '.csv')
stamp = time.strftime('%Y%m%d_%H%M')
zpath = ARTIFACTS / f'embeds_{stamp}.zip'

packed = 0
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for s in SCENES:
        d = Path(f'outputs/replica_{s}')
        if not d.exists():
            print(f'!! outputs/replica_{s} missing — cell 3 has not completed for {s}', flush=True)
            continue
        before = packed
        for f in sorted(d.iterdir()):
            if f.suffix in KEEP and f.stat().st_size < 300e6:
                z.write(f, f'outputs/replica_{s}/{f.name}')
                packed += 1
                print(f'{f.name:<45}{f.stat().st_size/1e6:7.1f} MB')
        if packed == before:
            print(f'!! outputs/replica_{s} exists but held no .pt/.json/.csv artifacts', flush=True)

if packed == 0:
    zpath.unlink()
    raise RuntimeError('Packed 0 files — no pipeline outputs exist yet. '
                       'Re-run cell 3 and watch its per-stage exit lines; '
                       'nothing was written or downloaded.')

print(f'\n{packed} files -> {zpath}  ({zpath.stat().st_size/1e6:.1f} MB)')
print('also saved in Drive/ssnslam_colab/artifacts — or download below')
from google.colab import files
files.download(str(zpath))


## 5 · Back on your machine

Unzip into the repo root — the paths inside the archive already match
the layout the local tools expect:

```bash
# from the repo root
tar -xf embeds_YYYYMMDD_HHMM.zip     # or unzip
```

Then everything downstream runs locally in seconds, e.g.:

```bash
python -m vsa_cognitive_mapping.object_grounding \
    --dataset vsa_cognitive_mapping/configs/replica_room0.json \
    --gt-json outputs/replica_room0/gt_instances.json --merge-check
python -m vsa_cognitive_mapping.relational_recall
python -m vsa_cognitive_mapping.instance_recall
python tools/aggregate_replica_truth.py
```

**What is deliberately NOT in this notebook:** the VSA analysis itself.
It is numpy, it is fast, the artifacts are small, and keeping it local
means the batteries and inspectors iterate at the speed of thought
instead of a Colab round-trip.
